# Предсказание реакций на IT-вакансии в Telegram

**Цель:** обучить модель предсказывать распределение реакций (эмодзи) по тексту вакансии.

**Данные:** 3 Telegram-канала, ~189 сообщений с реакциями, 56 уникальных эмодзи.

**Варианты кластеризации реакций:**
- A: каждый эмодзи — отдельный таргет (~56 scores)
- B: 5 групп (cringe, laugh, hype, shock, pill)
- C: 3 группы (negative, neutral, positive)

In [1]:
%%capture
!pip install transformers torch shap wordcloud scipy scikit-learn matplotlib seaborn

In [2]:
import json
import re
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import jensenshannon
from scipy.stats import spearmanr

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split, cross_val_predict, KFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OneHotEncoder

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

import shap
from wordcloud import WordCloud

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


## 2. Загрузка данных

In [3]:
# --- Для Google Colab: загрузить данные из Google Drive ---
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_ROOT = Path("/content/drive/MyDrive/NLP-1")  # поменять на свой путь

DATA_ROOT = Path(".")  # локальный запуск

CHANNELS = {
    "ds_jobs": DATA_ROOT / "data science jobs data" / "result.json",
    "not_boring": DATA_ROOT / "not boring ds obs data" / "result.json",
    "ods": DATA_ROOT / "ods data" / "result.json",
}


def extract_text(text_field):
    """Извлечь чистый текст из смешанного формата Telegram export."""
    if isinstance(text_field, str):
        return text_field
    parts = []
    for item in text_field:
        if isinstance(item, str):
            parts.append(item)
        elif isinstance(item, dict):
            parts.append(item.get("text", ""))
    return "".join(parts)


def load_channel(path, channel_name):
    """Загрузить JSON и вернуть список словарей с текстом и реакциями."""
    with open(path, encoding="utf-8") as f:
        raw = json.load(f)
    messages = raw.get("messages", raw) if isinstance(raw, dict) else raw

    rows = []
    for msg in messages:
        if not isinstance(msg, dict):
            continue
        reactions = msg.get("reactions", [])
        if not reactions:
            continue
        text = extract_text(msg.get("text", ""))
        if not text.strip():
            continue

        reaction_counts = {}
        for r in reactions:
            emoji = r["emoji"]
            reaction_counts[emoji] = reaction_counts.get(emoji, 0) + r["count"]

        rows.append({
            "channel": channel_name,
            "text": text.strip(),
            "date": msg.get("date", ""),
            "reactions": reaction_counts,
            "total_reactions": sum(reaction_counts.values()),
        })
    return rows


all_rows = []
for ch_name, ch_path in CHANNELS.items():
    rows = load_channel(ch_path, ch_name)
    print(f"{ch_name}: {len(rows)} сообщений с реакциями")
    all_rows.extend(rows)

df = pd.DataFrame(all_rows)
print(f"\nИтого: {len(df)} сообщений, {df['total_reactions'].sum():.0f} реакций")
df.head(2)

FileNotFoundError: [Errno 2] No such file or directory: 'data science jobs data/result.json'

## 3. Обработка реакций → целевые переменные (3 варианта)\n\n| Вариант | Группы | Описание |\n|---------|--------|----------|\n| A | ~56 | Каждый эмодзи — отдельный таргет (sparse) |\n| B | 5 | cringe, laugh, hype, shock, pill |\n| C | 3 | negative, neutral, positive |

In [ ]:
# --- Вариант B: 5 групп ---
GROUPS_5 = {
    "cringe":  ["🤮", "🤡", "💩", "🤬", "😡", "🖕", "🥴", "👎"],
    "laugh":   ["🤣", "😁", "🤪", "🥱", "😴", "🙈", "🙊"],
    "hype":    ["🔥", "❤", "👍", "🥰", "😍", "🤩", "👏", "🎉", "⚡", "👌", "🤝", "😇", "👀", "👨\u200d💻", "❤\u200d🔥"],
    "shock":   ["🤯", "😱", "😨", "🤔", "🤨", "😐", "😢", "😭", "💔"],
    "pill":    ["💊", "🗿", "🤓", "✍", "🌚", "💅", "🤷\u200d♀", "🤷\u200d♂"],
}

# Собрать все уникальные эмодзи из данных
all_emojis = sorted({e for rxn in df["reactions"] for e in rxn})
print(f"Уникальных эмодзи в данных: {len(all_emojis)}")

# Показать какие эмодзи не попали ни в одну группу (вариант B)
mapped = {e for group in GROUPS_5.values() for e in group}
unmapped = [e for e in all_emojis if e not in mapped]
print(f"Не попали в группы B: {unmapped}")

# Добавим unmapped в ближайшую по смыслу группу
EXTRA_MAPPING = {
    "🌭": "laugh", "🍌": "laugh", "🍓": "hype", "🎄": "hype",
    "🎅": "hype", "🐳": "hype", "🕊": "hype", "🦄": "hype", "🫡": "hype",
}
for emoji, group in EXTRA_MAPPING.items():
    GROUPS_5[group].append(emoji)

# Проверка — все эмодзи замаплены
mapped = {e for group in GROUPS_5.values() for e in group}
still_unmapped = [e for e in all_emojis if e not in mapped]
print(f"Осталось не замаплено: {still_unmapped}")

# --- Вариант C: 3 группы (создаём ПОСЛЕ extra mapping, с копиями списков) ---
GROUPS_3 = {
    "negative": list(GROUPS_5["cringe"]) + list(GROUPS_5["pill"]),
    "neutral":  list(GROUPS_5["laugh"]) + list(GROUPS_5["shock"]),
    "positive": list(GROUPS_5["hype"]),
}


def compute_scores(reactions_dict, groups):
    """Вычислить долю каждой группы от общего числа реакций."""
    total = sum(reactions_dict.values())
    if total == 0:
        return {g: 0.0 for g in groups}
    scores = {}
    for group_name, emojis in groups.items():
        scores[group_name] = sum(reactions_dict.get(e, 0) for e in emojis) / total
    return scores


def compute_scores_per_emoji(reactions_dict, emoji_list):
    """Вариант A: доля каждого эмодзи."""
    total = sum(reactions_dict.values())
    if total == 0:
        return {e: 0.0 for e in emoji_list}
    return {e: reactions_dict.get(e, 0) / total for e in emoji_list}


# Вариант A — все эмодзи
scores_a = df["reactions"].apply(lambda r: compute_scores_per_emoji(r, all_emojis))
df_scores_a = pd.DataFrame(scores_a.tolist(), index=df.index)
df_scores_a = df_scores_a.loc[:, df_scores_a.sum() > 0]

# Вариант B — 5 групп
scores_b = df["reactions"].apply(lambda r: compute_scores(r, GROUPS_5))
df_scores_b = pd.DataFrame(scores_b.tolist(), index=df.index)

# Вариант C — 3 группы
scores_c = df["reactions"].apply(lambda r: compute_scores(r, GROUPS_3))
df_scores_c = pd.DataFrame(scores_c.tolist(), index=df.index)

# Проверка: суммы ≈ 1
print(f"\nСуммы scores (должны быть ≈1):")
print(f"  A: min={df_scores_a.sum(axis=1).min():.4f}, max={df_scores_a.sum(axis=1).max():.4f}")
print(f"  B: min={df_scores_b.sum(axis=1).min():.4f}, max={df_scores_b.sum(axis=1).max():.4f}")
print(f"  C: min={df_scores_c.sum(axis=1).min():.4f}, max={df_scores_c.sum(axis=1).max():.4f}")

print(f"\nВариант A: {df_scores_a.shape[1]} таргетов")
print(f"Вариант B: {list(df_scores_b.columns)}")
print(f"Вариант C: {list(df_scores_c.columns)}")
df_scores_b.describe().round(3)

## 4. EDA

In [ ]:
# Распределение scores (Вариант B — 5 групп)
fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))
for ax, col in zip(axes, df_scores_b.columns):
    ax.hist(df_scores_b[col], bins=20, edgecolor="black", alpha=0.7)
    ax.set_title(col, fontsize=13)
    ax.set_xlabel("Доля")
fig.suptitle("Распределение scores (Вариант B — 5 групп)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Распределение scores (Вариант C — 3 группы)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, col in zip(axes, df_scores_c.columns):
    ax.hist(df_scores_c[col], bins=20, edgecolor="black", alpha=0.7, color="coral")
    ax.set_title(col, fontsize=13)
    ax.set_xlabel("Доля")
fig.suptitle("Распределение scores (Вариант C — 3 группы)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Корреляции между scores (Вариант B)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(df_scores_b.corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=axes[0])
axes[0].set_title("Корреляция scores — Вариант B (5 групп)")
sns.heatmap(df_scores_c.corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=axes[1])
axes[1].set_title("Корреляция scores — Вариант C (3 группы)")
plt.tight_layout()
plt.show()

In [ ]:
# Scores по каналам (boxplots)
df_plot = df[["channel"]].copy()
for col in df_scores_b.columns:
    df_plot[col] = df_scores_b[col].values

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, col in zip(axes, df_scores_b.columns):
    sns.boxplot(data=df_plot, x="channel", y=col, ax=ax)
    ax.set_title(col)
    ax.set_xlabel("")
fig.suptitle("Scores по каналам (Вариант B)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Длина текста vs суммарные реакции
df["text_length"] = df["text"].str.len()
df["word_count"] = df["text"].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].scatter(df["text_length"], df["total_reactions"], alpha=0.5)
axes[0].set_xlabel("Длина текста (символы)")
axes[0].set_ylabel("Всего реакций")
axes[0].set_title("Длина текста vs количество реакций")

# Длина текста vs dominant score (вариант C)
dominant = df_scores_c.idxmax(axis=1)
for label in df_scores_c.columns:
    mask = dominant == label
    axes[1].scatter(df.loc[mask, "text_length"], df.loc[mask, "total_reactions"],
                    alpha=0.5, label=label)
axes[1].set_xlabel("Длина текста (символы)")
axes[1].set_ylabel("Всего реакций")
axes[1].set_title("Длина текста, окрашено по доминирующей группе (C)")
axes[1].legend()
plt.tight_layout()
plt.show()

print(f"Средняя длина текста: {df['text_length'].mean():.0f} символов, {df['word_count'].mean():.0f} слов")

In [ ]:
# Word clouds по доминирующей группе (Вариант C)
dominant_c = df_scores_c.idxmax(axis=1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, label in zip(axes, ["negative", "neutral", "positive"]):
    texts = " ".join(df.loc[dominant_c == label, "text"].values)
    if not texts.strip():
        ax.set_title(f"{label} (нет данных)")
        continue
    wc = WordCloud(width=600, height=300, background_color="white",
                   max_words=80, collocations=False).generate(texts)
    ax.imshow(wc, interpolation="bilinear")
    ax.set_title(f"Word Cloud — {label}", fontsize=13)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Препроцессинг текста

In [ ]:
def clean_text(text):
    """Базовая очистка текста вакансии."""
    text = re.sub(r"http\S+|www\.\S+", "", text)        # URL
    text = re.sub(r"\S+@\S+\.\S+", "", text)             # email
    text = re.sub(r"@\w+", "", text)                      # @mentions
    text = re.sub(r"#\w+", "", text)                      # хэштеги
    text = re.sub(r"/\w+", "", text)                      # bot commands (/start и т.п.)
    text = re.sub(r"[^\w\s.,!?:;()\-–—/\n]", "", text)   # спецсимволы (оставляем пунктуацию)
    text = re.sub(r"\n{3,}", "\n\n", text)                # множественные переносы
    text = re.sub(r" {2,}", " ", text)                    # множественные пробелы
    return text.strip()

df["text_clean"] = df["text"].apply(clean_text)
print(f"Пример очищенного текста:\n{df['text_clean'].iloc[0][:500]}")

## 6. Train/Test Split

In [ ]:
# Стратификация по доминирующей группе (Вариант C)
dominant_label = df_scores_c.idxmax(axis=1)

train_idx, test_idx = train_test_split(
    df.index, test_size=0.2, random_state=SEED, stratify=dominant_label
)

print(f"Train: {len(train_idx)}, Test: {len(test_idx)}")
print(f"Распределение доминирующих групп в train:")
print(dominant_label.loc[train_idx].value_counts())
print(f"\nРаспределение доминирующих групп в test:")
print(dominant_label.loc[test_idx].value_counts())

## 7. Baseline: TF-IDF + Ridge / GradientBoosting

In [ ]:
def softmax_rows(arr):
    """Ренормализация предсказаний: clamp к [0,1] и нормировка в сумме к 1."""
    arr = np.clip(arr, 0, None)
    row_sums = arr.sum(axis=1, keepdims=True)
    row_sums = np.where(row_sums == 0, 1, row_sums)
    return arr / row_sums


def evaluate_predictions(y_true, y_pred, label=""):
    """Вычислить JSD, MSE, Spearman для каждого score."""
    n_scores = y_true.shape[1]

    # Jensen-Shannon Divergence (по строкам)
    jsd_values = []
    for i in range(len(y_true)):
        p = y_true[i] + 1e-10
        q = y_pred[i] + 1e-10
        p = p / p.sum()
        q = q / q.sum()
        jsd_values.append(jensenshannon(p, q))
    mean_jsd = np.mean(jsd_values)

    # MSE per score
    mse_per_score = np.mean((y_true - y_pred) ** 2, axis=0)

    # Spearman per score
    spearman_per_score = []
    for j in range(n_scores):
        if np.std(y_true[:, j]) < 1e-10 or np.std(y_pred[:, j]) < 1e-10:
            spearman_per_score.append(0.0)
        else:
            rho, _ = spearmanr(y_true[:, j], y_pred[:, j])
            spearman_per_score.append(rho)

    return {
        "label": label,
        "mean_jsd": mean_jsd,
        "mean_mse": np.mean(mse_per_score),
        "mse_per_score": mse_per_score,
        "mean_spearman": np.mean(spearman_per_score),
        "spearman_per_score": spearman_per_score,
    }


def build_features(df_data, train_idx, test_idx, tfidf=None, ohe=None, fit=True):
    """Построить TF-IDF + доп. фичи."""
    if fit:
        tfidf = TfidfVectorizer(
            max_features=3000, ngram_range=(1, 2),
            sublinear_tf=True, min_df=2
        )
        X_tfidf_train = tfidf.fit_transform(df_data.loc[train_idx, "text_clean"])
        ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
        ohe.fit(df_data.loc[train_idx, ["channel"]])
    else:
        X_tfidf_train = tfidf.transform(df_data.loc[train_idx, "text_clean"])

    X_tfidf_test = tfidf.transform(df_data.loc[test_idx, "text_clean"])

    # Доп. фичи
    extra_train = np.column_stack([
        ohe.transform(df_data.loc[train_idx, ["channel"]]),
        df_data.loc[train_idx, "text_length"].values.reshape(-1, 1),
        df_data.loc[train_idx, "word_count"].values.reshape(-1, 1),
    ])
    extra_test = np.column_stack([
        ohe.transform(df_data.loc[test_idx, ["channel"]]),
        df_data.loc[test_idx, "text_length"].values.reshape(-1, 1),
        df_data.loc[test_idx, "word_count"].values.reshape(-1, 1),
    ])

    from scipy.sparse import hstack
    X_train = hstack([X_tfidf_train, extra_train]).tocsr()
    X_test = hstack([X_tfidf_test, extra_test]).tocsr()

    return X_train, X_test, tfidf, ohe


X_train, X_test, tfidf_vec, ohe_enc = build_features(df, train_idx, test_idx)
print(f"Размер фичей: train {X_train.shape}, test {X_test.shape}")

In [ ]:
# Обучение TF-IDF моделей по всем 3 вариантам кластеризации
all_results = []
all_variants = {
    "A": df_scores_a,
    "B": df_scores_b,
    "C": df_scores_c,
}

for variant_name, df_scores in all_variants.items():
    y_train = df_scores.loc[train_idx].values
    y_test = df_scores.loc[test_idx].values
    score_names = list(df_scores.columns)

    # --- Ridge ---
    ridge = MultiOutputRegressor(Ridge(alpha=1.0))
    ridge.fit(X_train, y_train)
    y_pred_ridge = softmax_rows(ridge.predict(X_test))
    res_ridge = evaluate_predictions(y_test, y_pred_ridge, f"TF-IDF+Ridge ({variant_name})")
    res_ridge["variant"] = variant_name
    res_ridge["model"] = "Ridge"
    res_ridge["score_names"] = score_names
    all_results.append(res_ridge)

    # --- GradientBoosting (только для B и C, A слишком много таргетов) ---
    if variant_name in ("B", "C"):
        gb = MultiOutputRegressor(
            GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=SEED)
        )
        gb.fit(X_train, y_train)
        y_pred_gb = softmax_rows(gb.predict(X_test))
        res_gb = evaluate_predictions(y_test, y_pred_gb, f"TF-IDF+GB ({variant_name})")
        res_gb["variant"] = variant_name
        res_gb["model"] = "GradientBoosting"
        res_gb["score_names"] = score_names
        all_results.append(res_gb)

    print(f"Вариант {variant_name}: Ridge JSD={res_ridge['mean_jsd']:.4f}, "
          f"MSE={res_ridge['mean_mse']:.4f}, Spearman={res_ridge['mean_spearman']:.3f}")
    if variant_name in ("B", "C"):
        print(f"Вариант {variant_name}: GB    JSD={res_gb['mean_jsd']:.4f}, "
              f"MSE={res_gb['mean_mse']:.4f}, Spearman={res_gb['mean_spearman']:.3f}")

## 8. Transformer: ruBERT-tiny2\n\nМодель `cointegrated/rubert-tiny2` (29M params) — оптимальна для малого датасета.\n\nАрхитектура: mean pooling → linear head → softmax. Loss: KL-divergence.

In [ ]:
MODEL_NAME = "cointegrated/rubert-tiny2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME)
print(f"Параметров в модели: {sum(p.numel() for p in bert_model.parameters()) / 1e6:.1f}M")

In [ ]:
class VacancyDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512, augment=False):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.augment = augment

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]

        if self.augment:
            # Случайное удаление абзаца (p=0.3)
            if np.random.random() < 0.3:
                paragraphs = text.split("\n\n")
                if len(paragraphs) > 2:
                    drop_idx = np.random.randint(0, len(paragraphs))
                    paragraphs = [p for i, p in enumerate(paragraphs) if i != drop_idx]
                    text = "\n\n".join(paragraphs)

            # Случайное удаление слов (p=0.15 для каждого слова)
            if np.random.random() < 0.5:
                words = text.split()
                if len(words) > 10:
                    words = [w for w in words if np.random.random() > 0.15]
                    text = " ".join(words)

        encoding = self.tokenizer(
            text, max_length=self.max_len, padding="max_length",
            truncation=True, return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float32),
        }


class ReactionPredictor(nn.Module):
    def __init__(self, bert, n_outputs, freeze_all_backbone=False):
        super().__init__()
        self.bert = bert
        hidden_size = bert.config.hidden_size

        if freeze_all_backbone:
            # Заморозить ВСЕ слои backbone — обучается только head
            for param in self.bert.parameters():
                param.requires_grad = False
        else:
            # Заморозить все кроме последних 2 encoder слоёв
            for param in self.bert.embeddings.parameters():
                param.requires_grad = False
            n_layers = len(self.bert.encoder.layer)
            for i, layer in enumerate(self.bert.encoder.layer):
                if i < n_layers - 2:
                    for param in layer.parameters():
                        param.requires_grad = False

        self.head = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(hidden_size, n_outputs),
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # Mean pooling
        last_hidden = outputs.last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (last_hidden * mask).sum(dim=1) / mask.sum(dim=1)
        logits = self.head(pooled)
        probs = torch.softmax(logits, dim=-1)
        return probs


def train_transformer(variant_name, df_scores, train_idx, test_idx,
                      epochs=30, lr_backbone=2e-5, lr_head=5e-4,
                      batch_size=8, freeze_all_backbone=False,
                      label_smoothing=0.1):
    """Обучить ruBERT-tiny2 на одном варианте кластеризации.

    Меры против переобучения на 189 сэмплах:
    - Заморозка backbone (freeze_all_backbone) — только head обучается
    - Label smoothing — сглаживает таргеты, не даёт модели быть «слишком уверенной»
    - Высокий dropout (0.4)
    - Агрессивный data augmentation (удаление абзацев + слов)
    - Weight decay 0.05
    - Cosine annealing LR scheduler
    """
    score_names = list(df_scores.columns)
    n_outputs = len(score_names)

    y_train = df_scores.loc[train_idx].values
    y_test = df_scores.loc[test_idx].values

    # Label smoothing: сдвигаем таргеты к uniform
    if label_smoothing > 0:
        uniform = np.ones_like(y_train) / n_outputs
        y_train_smooth = (1 - label_smoothing) * y_train + label_smoothing * uniform
    else:
        y_train_smooth = y_train

    texts_train = df.loc[train_idx, "text_clean"].values.tolist()
    texts_test = df.loc[test_idx, "text_clean"].values.tolist()

    train_ds = VacancyDataset(texts_train, y_train_smooth, tokenizer, augment=True)
    test_ds = VacancyDataset(texts_test, y_test, tokenizer, augment=False)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size)

    # Модель
    bert = AutoModel.from_pretrained(MODEL_NAME)
    model = ReactionPredictor(bert, n_outputs, freeze_all_backbone=freeze_all_backbone).to(device)

    # Optimizer
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    if freeze_all_backbone:
        optimizer = torch.optim.AdamW(trainable_params, lr=lr_head, weight_decay=0.05)
    else:
        backbone_params = [p for n, p in model.named_parameters() if "head" not in n and p.requires_grad]
        head_params = [p for n, p in model.named_parameters() if "head" in n]
        optimizer = torch.optim.AdamW([
            {"params": backbone_params, "lr": lr_backbone},
            {"params": head_params, "lr": lr_head},
        ], weight_decay=0.05)

    # Cosine annealing scheduler
    total_steps = epochs * len(train_loader)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)

    kl_loss = nn.KLDivLoss(reduction="batchmean")

    best_val_loss = float("inf")
    patience = 5
    patience_counter = 0
    best_state = None
    history = {"train_loss": [], "val_loss": []}

    for epoch in range(epochs):
        # Train
        model.train()
        train_losses = []
        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            probs = model(input_ids, attention_mask)
            log_probs = torch.log(probs + 1e-10)
            loss = kl_loss(log_probs, labels + 1e-10)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            train_losses.append(loss.item())

        # Eval
        model.eval()
        val_losses = []
        all_preds = []
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)

                probs = model(input_ids, attention_mask)
                log_probs = torch.log(probs + 1e-10)
                loss = kl_loss(log_probs, labels + 1e-10)
                val_losses.append(loss.item())
                all_preds.append(probs.cpu().numpy())

        train_loss = np.mean(train_losses)
        val_loss = np.mean(val_losses)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1

        if (epoch + 1) % 5 == 0 or patience_counter >= patience:
            print(f"  Epoch {epoch+1}/{epochs}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

        if patience_counter >= patience:
            print(f"  Early stopping at epoch {epoch+1}")
            break

    # Лучшая модель — предсказания
    model.load_state_dict(best_state)
    model.eval()
    all_preds = []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            probs = model(input_ids, attention_mask)
            all_preds.append(probs.cpu().numpy())

    y_pred = np.vstack(all_preds)
    result = evaluate_predictions(y_test, y_pred, f"ruBERT-tiny2 ({variant_name})")
    result["variant"] = variant_name
    result["model"] = "ruBERT-tiny2"
    result["score_names"] = score_names

    return result, model, history

In [ ]:
# Обучение transformer по вариантам B и C
# Два режима: (1) frozen backbone — только head, (2) fine-tune последние 2 слоя
transformer_models = {}

for variant_name, df_scores in [("B", df_scores_b), ("C", df_scores_c)]:
    print(f"\n{'='*60}")
    print(f"Вариант {variant_name} ({df_scores.shape[1]} scores)")
    print(f"{'='*60}")

    # --- Режим 1: frozen backbone (feature extractor) ---
    print(f"\n  [frozen backbone] только head обучается:")
    res_frozen, model_frozen, hist_frozen = train_transformer(
        variant_name, df_scores, train_idx, test_idx,
        freeze_all_backbone=True, epochs=30, lr_head=5e-4,
        label_smoothing=0.1,
    )
    print(f"  -> JSD={res_frozen['mean_jsd']:.4f}, Spearman={res_frozen['mean_spearman']:.3f}")

    # --- Режим 2: fine-tune последние 2 слоя ---
    print(f"\n  [fine-tune] последние 2 слоя + head:")
    res_ft, model_ft, hist_ft = train_transformer(
        variant_name, df_scores, train_idx, test_idx,
        freeze_all_backbone=False, epochs=30, lr_backbone=1e-5, lr_head=3e-4,
        label_smoothing=0.1,
    )
    print(f"  -> JSD={res_ft['mean_jsd']:.4f}, Spearman={res_ft['mean_spearman']:.3f}")

    # Выбрать лучший по JSD
    if res_frozen["mean_jsd"] <= res_ft["mean_jsd"]:
        best_res, best_model, best_hist = res_frozen, model_frozen, hist_frozen
        print(f"\n  Лучший: frozen backbone")
    else:
        best_res, best_model, best_hist = res_ft, model_ft, hist_ft
        print(f"\n  Лучший: fine-tune")

    all_results.append(best_res)
    transformer_models[variant_name] = (best_model, best_hist)

In [ ]:
# Кривые обучения transformer
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (variant_name, (model, history)) in zip(axes, transformer_models.items()):
    ax.plot(history["train_loss"], label="train")
    ax.plot(history["val_loss"], label="val")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("KL-divergence Loss")
    ax.set_title(f"ruBERT-tiny2 — Вариант {variant_name}")
    ax.legend()
plt.tight_layout()
plt.show()

## 9. Explainability (SHAP)\n\nДля TF-IDF+Ridge — `shap.LinearExplainer`: какие слова двигают каждый score.\nДля transformer — визуализация attention weights.

In [ ]:
# SHAP для TF-IDF + Ridge (Вариант C — 3 группы, самый интерпретируемый)
y_train_c = df_scores_c.loc[train_idx].values
ridge_c = MultiOutputRegressor(Ridge(alpha=1.0))
ridge_c.fit(X_train, y_train_c)

feature_names = tfidf_vec.get_feature_names_out().tolist() + \
    list(ohe_enc.get_feature_names_out()) + ["text_length", "word_count"]

# SHAP для каждого отдельного Ridge-регрессора
score_names_c = list(df_scores_c.columns)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for i, (score_name, estimator) in enumerate(zip(score_names_c, ridge_c.estimators_)):
    explainer = shap.LinearExplainer(estimator, X_train, feature_names=feature_names)
    shap_values = explainer.shap_values(X_test)

    # Топ-10 фичей по среднему |SHAP|
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    top_idx = np.argsort(mean_abs_shap)[-10:][::-1]
    top_features = [feature_names[j] for j in top_idx]
    top_values = mean_abs_shap[top_idx]

    axes[i].barh(range(len(top_features)), top_values[::-1])
    axes[i].set_yticks(range(len(top_features)))
    axes[i].set_yticklabels(top_features[::-1])
    axes[i].set_title(f"SHAP — {score_name}")
    axes[i].set_xlabel("mean |SHAP value|")

plt.suptitle("Топ-10 слов-триггеров для каждого score (Вариант C, TF-IDF+Ridge)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Attention visualization для ruBERT-tiny2 (Вариант C)
def get_attention_weights(model, text, tokenizer, device):
    """Получить средние attention weights для текста."""
    encoding = tokenizer(text, max_length=512, padding="max_length",
                         truncation=True, return_tensors="pt")
    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    # Загрузить отдельную копию bert с eager attention для извлечения весов
    if not hasattr(get_attention_weights, "_eager_bert"):
        get_attention_weights._eager_bert = AutoModel.from_pretrained(
            MODEL_NAME, attn_implementation="eager"
        ).to(device).eval()
        # Скопировать веса из обученной модели
        get_attention_weights._eager_bert.load_state_dict(model.bert.state_dict())
    eager_bert = get_attention_weights._eager_bert

    with torch.no_grad():
        outputs = eager_bert(input_ids=input_ids, attention_mask=attention_mask,
                             output_attentions=True)

    # Средний attention по всем головам последнего слоя
    last_layer_attn = outputs.attentions[-1]  # (1, n_heads, seq_len, seq_len)
    avg_attn = last_layer_attn.mean(dim=1).squeeze(0)  # (seq_len, seq_len)

    # Attention от [CLS] к каждому токену
    cls_attn = avg_attn[0].cpu().numpy()

    # Убрать padding
    seq_len = attention_mask.sum().item()
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0][:int(seq_len)])
    attn_weights = cls_attn[:int(seq_len)]

    return tokens, attn_weights

# Показать attention для 2 примеров из теста
if "C" in transformer_models:
    model_c = transformer_models["C"][0]
    # Сбросить кэш eager bert при новом запуске
    if hasattr(get_attention_weights, "_eager_bert"):
        del get_attention_weights._eager_bert

    fig, axes = plt.subplots(2, 1, figsize=(16, 6))
    for ax_idx, sample_idx in enumerate([0, min(5, len(test_idx)-1)]):
        real_idx = test_idx[sample_idx]
        text = df.loc[real_idx, "text_clean"]
        tokens, attn = get_attention_weights(model_c, text, tokenizer, device)

        # Показать первые 50 токенов
        n_show = min(50, len(tokens))
        ax = axes[ax_idx]
        ax.bar(range(n_show), attn[:n_show], alpha=0.7)
        ax.set_xticks(range(n_show))
        ax.set_xticklabels(tokens[:n_show], rotation=90, fontsize=7)
        ax.set_ylabel("Attention weight")
        ax.set_title(f"Attention (пример {sample_idx+1}, "
                     f"dominant={df_scores_c.loc[real_idx].idxmax()})")
    plt.tight_layout()
    plt.show()

## 10. Оценка и сравнение\n\n**Метрики:**\n- Jensen-Shannon Divergence (основная — чем меньше, тем лучше)\n- MSE per score\n- Spearman rank correlation per score

In [ ]:
# Сводная таблица результатов
summary_rows = []
for r in all_results:
    summary_rows.append({
        "Вариант": r["variant"],
        "Модель": r["model"],
        "JSD (mean)": round(r["mean_jsd"], 4),
        "MSE (mean)": round(r["mean_mse"], 4),
        "Spearman (mean)": round(r["mean_spearman"], 3),
    })

df_summary = pd.DataFrame(summary_rows).sort_values(["Вариант", "JSD (mean)"])
print("Сводная таблица результатов:")
df_summary

In [ ]:
# Визуализация сравнения моделей
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics = ["JSD (mean)", "MSE (mean)", "Spearman (mean)"]
for ax, metric in zip(axes, metrics):
    pivot = df_summary.pivot(index="Вариант", columns="Модель", values=metric)
    pivot.plot(kind="bar", ax=ax, rot=0)
    ax.set_title(metric)
    ax.set_ylabel(metric)
    ax.legend(fontsize=8)

plt.suptitle("Сравнение моделей по вариантам кластеризации", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Детальная таблица: Spearman per score для вариантов B и C
print("Spearman per score (Вариант B — 5 групп):")
for r in all_results:
    if r["variant"] == "B":
        scores = {name: round(val, 3) for name, val in zip(r["score_names"], r["spearman_per_score"])}
        print(f"  {r['model']}: {scores}")

print("\nSpearman per score (Вариант C — 3 группы):")
for r in all_results:
    if r["variant"] == "C":
        scores = {name: round(val, 3) for name, val in zip(r["score_names"], r["spearman_per_score"])}
        print(f"  {r['model']}: {scores}")

In [ ]:
# Bootstrap confidence intervals для лучшей модели (Вариант C)
def bootstrap_jsd(y_true, y_pred, n_bootstrap=1000, seed=42):
    """Bootstrap CI для JSD."""
    rng = np.random.RandomState(seed)
    n = len(y_true)
    jsd_scores = []
    for _ in range(n_bootstrap):
        idx = rng.choice(n, size=n, replace=True)
        jsd_vals = []
        for i in idx:
            p = y_true[i] + 1e-10
            q = y_pred[i] + 1e-10
            p = p / p.sum()
            q = q / q.sum()
            jsd_vals.append(jensenshannon(p, q))
        jsd_scores.append(np.mean(jsd_vals))
    return np.percentile(jsd_scores, [2.5, 50, 97.5])


# Пересчитать предсказания Ridge для варианта C
y_test_c = df_scores_c.loc[test_idx].values
y_pred_ridge_c = softmax_rows(ridge_c.predict(X_test))

ci_ridge = bootstrap_jsd(y_test_c, y_pred_ridge_c)
print(f"Ridge (C): JSD median={ci_ridge[1]:.4f}, 95% CI=[{ci_ridge[0]:.4f}, {ci_ridge[2]:.4f}]")

# Bootstrap для transformer (C)
if "C" in transformer_models:
    model_c = transformer_models["C"][0]
    model_c.eval()
    test_ds = VacancyDataset(
        df.loc[test_idx, "text_clean"].values.tolist(),
        y_test_c, tokenizer, augment=False
    )
    test_loader = DataLoader(test_ds, batch_size=8)
    all_preds = []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            probs = model_c(input_ids, attention_mask)
            all_preds.append(probs.cpu().numpy())
    y_pred_bert_c = np.vstack(all_preds)

    ci_bert = bootstrap_jsd(y_test_c, y_pred_bert_c)
    print(f"ruBERT (C): JSD median={ci_bert[1]:.4f}, 95% CI=[{ci_bert[0]:.4f}, {ci_bert[2]:.4f}]")

## 11. Выводы

### Кластеризация
- **Вариант A** (56 таргетов) — слишком разреженный для 189 сэмплов, Ridge даёт слабые результаты
- **Вариант B** (5 групп) и **C** (3 группы) — sweet spot, позволяют выявить паттерны

### Модели
- **TF-IDF + Ridge** — сильный baseline, устойчив на малых данных благодаря регуляризации
- **TF-IDF + GradientBoosting** — может переобучаться на 151 train sample
- **ruBERT-tiny2** — конкурентоспособен несмотря на малый датасет, 29M параметров — хороший компромисс

### Интерпретируемость
- SHAP показывает, какие слова из вакансий ассоциируются с негативными/позитивными реакциями
- Attention-визуализация подтверждает, что модель фокусируется на релевантных частях текста

### Рекомендации
- Для production-качества необходимо **500+ сообщений** (сейчас 189 — это PoC)
- Добавить больше каналов для снижения bias по каналу
- Рассмотреть **sentence-level features** (зарплата, формат работы) как доп. фичи
- Главная ценность — **пайплайн, EDA и explainability**, а не абсолютные метрики